# Занятие 2. Переобучение и регуляризация

## Цели занятия:
- Наглядно продемонстрировать эффект переобучения (overfitting)
- Изучить методы регуляризации: Dropout, BatchNorm, L2
- Освоить аугментацию данных через torchvision.transforms
- Сравнить эффективность методов на практике

---

## Шаг 1. Подготовка урезанного датасета

**Важно:** Для демонстрации переобучения используем только 5% данных!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import random

# Fix random seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Standard transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Full dataset
full_dataset = datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)

# Subset 5% for overfitting demo
subset_size = int(len(full_dataset) * 0.05)
indices = torch.randperm(len(full_dataset))[:subset_size]
small_dataset = Subset(full_dataset, indices)

# Create loaders
small_loader = DataLoader(small_dataset, batch_size=64, shuffle=True)

# Full test set for validation
test_dataset = datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Total training samples: {len(full_dataset)}")
print(f"Subset samples (5%): {len(small_dataset)}")
print(f"Test samples: {len(test_dataset)}")

---
## Шаг 2. Базовая модель (без регуляризации)

**Задание:** Создайте модель с большой ёмкостью для демонстрации переобучения.

In [ ]:
class OverfitNet(nn.Module):
    """Large capacity model to demonstrate overfitting."""
    
    def __init__(self):
        super(OverfitNet, self).__init__()
        # Large capacity model for small data
        self.conv1 = nn.Conv2d(1, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.fc1 = nn.Linear(128 * 7 * 7, 512)
        self.fc2 = nn.Linear(512, 10)
        self.pool = nn.MaxPool2d(2, 2)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 128 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model_overfit = OverfitNet().to(device)
total_params = sum(p.numel() for p in model_overfit.parameters())
print(f"OverfitNet parameters: {total_params:,}")

---
## Эксперимент 1: Демонстрация переобучения

In [ ]:
from tqdm import tqdm

def train_and_validate(model, train_loader, val_loader, epochs=20, lr=0.01, weight_decay=0.0):
    """Train model and return history."""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
        
        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        history['train_loss'].append(train_loss / len(train_loader))
        history['train_acc'].append(100 * train_correct / train_total)
        history['val_loss'].append(val_loss / len(val_loader))
        history['val_acc'].append(100 * val_correct / val_total)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}: Train Acc={history['train_acc'][-1]:.1f}%, "
                  f"Val Acc={history['val_acc'][-1]:.1f}%")
    
    return history

In [ ]:
# Experiment 1: No regularization
print("="*50)
print("Experiment 1: No Regularization (Overfitting)")
print("="*50)

set_seed(42)
model_overfit = OverfitNet().to(device)
history_overfit = train_and_validate(
    model_overfit, small_loader, test_loader, epochs=20, lr=0.01
)

print(f"\nFinal: Train Acc={history_overfit['train_acc'][-1]:.1f}%, "
      f"Val Acc={history_overfit['val_acc'][-1]:.1f}%")
print(f"Gap: {history_overfit['train_acc'][-1] - history_overfit['val_acc'][-1]:.1f}%")

---
## Шаг 4. Эксперимент 2: Dropout и Weight Decay (L2)

In [ ]:
class RegularizedNet(nn.Module):
    """Model with BatchNorm and Dropout for regularization."""
    
    def __init__(self):
        super(RegularizedNet, self).__init__()
        
        self.conv1 = nn.Conv2d(1, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)  # Batch Normalization
        
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        
        self.fc1 = nn.Linear(128 * 7 * 7, 512)
        self.dropout = nn.Dropout(0.5)  # Dropout
        self.fc2 = nn.Linear(512, 10)
        
        self.pool = nn.MaxPool2d(2, 2)
    
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(-1, 128 * 7 * 7)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [ ]:
# Experiment 2: With regularization
print("="*50)
print("Experiment 2: BatchNorm + Dropout + L2 (weight_decay)")
print("="*50)

set_seed(42)
model_reg = RegularizedNet().to(device)
history_reg = train_and_validate(
    model_reg, small_loader, test_loader, epochs=20, lr=0.01, weight_decay=1e-4
)

print(f"\nFinal: Train Acc={history_reg['train_acc'][-1]:.1f}%, "
      f"Val Acc={history_reg['val_acc'][-1]:.1f}%")
print(f"Gap: {history_reg['train_acc'][-1] - history_reg['val_acc'][-1]:.1f}%")

---
## Шаг 5. Эксперимент 3: Аугментация данных

In [ ]:
# Transforms with Augmentation
train_transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Reload dataset with augmentation
aug_dataset = datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=train_transform_aug
)

# Apply same subset indices
small_aug_dataset = Subset(aug_dataset, indices)
small_aug_loader = DataLoader(small_aug_dataset, batch_size=64, shuffle=True)

In [ ]:
# Experiment 3: With augmentation
print("="*50)
print("Experiment 3: Augmentation + Regularization")
print("="*50)

set_seed(42)
model_aug = RegularizedNet().to(device)
history_aug = train_and_validate(
    model_aug, small_aug_loader, test_loader, epochs=20, lr=0.01, weight_decay=1e-4
)

print(f"\nFinal: Train Acc={history_aug['train_acc'][-1]:.1f}%, "
      f"Val Acc={history_aug['val_acc'][-1]:.1f}%")
print(f"Gap: {history_aug['train_acc'][-1] - history_aug['val_acc'][-1]:.1f}%")

---
## Шаг 6. Сравнение результатов

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Summary table
results = {
    'Experiment': ['Baseline (Overfit)', 'Dropout+BN+L2', 'Augmentation'],
    'Train Acc': [history_overfit['train_acc'][-1], history_reg['train_acc'][-1], history_aug['train_acc'][-1]],
    'Val Acc': [history_overfit['val_acc'][-1], history_reg['val_acc'][-1], history_aug['val_acc'][-1]],
    'Gap': [history_overfit['train_acc'][-1] - history_overfit['val_acc'][-1],
            history_reg['train_acc'][-1] - history_reg['val_acc'][-1],
            history_aug['train_acc'][-1] - history_aug['val_acc'][-1]]
}

df = pd.DataFrame(results)
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
print(df.to_string(index=False))

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot comparison
x = range(1, 21)

# Experiment 1
axes[0].plot(x, history_overfit['train_acc'], 'b-', label='Train', linewidth=2)
axes[0].plot(x, history_overfit['val_acc'], 'r--', label='Val', linewidth=2)
axes[0].fill_between(x, history_overfit['train_acc'], history_overfit['val_acc'], alpha=0.3, color='red')
axes[0].set_title('Baseline (Overfitting)', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy (%)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Experiment 2
axes[1].plot(x, history_reg['train_acc'], 'b-', label='Train', linewidth=2)
axes[1].plot(x, history_reg['val_acc'], 'r--', label='Val', linewidth=2)
axes[1].fill_between(x, history_reg['train_acc'], history_reg['val_acc'], alpha=0.3, color='red')
axes[1].set_title('Regularization (BN+Dropout+L2)', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Experiment 3
axes[2].plot(x, history_aug['train_acc'], 'b-', label='Train', linewidth=2)
axes[2].plot(x, history_aug['val_acc'], 'r--', label='Val', linewidth=2)
axes[2].fill_between(x, history_aug['train_acc'], history_aug['val_acc'], alpha=0.3, color='red')
axes[2].set_title('Regularization + Augmentation', fontsize=12)
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('regularization_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(results['Experiment']))
width = 0.35

bars1 = ax.bar(x - width/2, results['Train Acc'], width, label='Train Accuracy', color='steelblue')
bars2 = ax.bar(x + width/2, results['Val Acc'], width, label='Val Accuracy', color='coral')

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Impact of Regularization Methods', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(results['Experiment'], fontsize=10)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('regularization_bar.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Домашнее задание

1. **Подобрать оптимальный коэффициент weight_decay**
   - Попробуйте значения из ряда 10⁻³, 10⁻⁴, 10⁻⁵
   - Сравните результаты в таблице

2. **Попробовать комбинацию всех методов**
   - BatchNorm + Dropout + Augmentation + L2
   - Зафиксируйте лучший результат

3. **Заполнить таблицу результатов в отчёте**

---
## Контрольные вопросы

1. Почему accuracy на обучении может быть 100%, а на валидации низкой?
2. Как работает Dropout во время обучения и во время инференса?
3. Зачем нужна Batch Normalization перед активацией или после?
4. Как аугментация влияет на размер эффективного датасета?

In [ ]:
# Место для экспериментов с домашним заданием
# TODO: Проведите эксперименты с разными weight_decay